In [1]:
import pandas as pd 
import numpy as np
import sklearn
import pickle

In [ ]:
# Lineair model

def mlr_train(country_dfs):
    models = {}
    for country in country_dfs:
        model = sklearn.linear_model.LinearRegression()
        x = country_dfs[country]['train'].index.to_numpy().reshape(-1, 1)
        y = country_dfs[country]['train']
        model.fit(x, y)
        models[country] = model
    return(models) #dict van de modellen voor elk land

# Output van linear model
# De input benodigd hiervoor is het dict van alle gebruikte landen, hieruit wordt per land de test split gebruikt. Ook zijn de modellen gegenereerd door de voorgaande functie bijnodigd.
# Het resultaat is een dict van dataframes met de voorspelde waarde voor elk jaar in de test split.

def mlr_test(country_dfs, models):
    predictions = {}
    for country in country_dfs:
        model = models[country]
        x = country_dfs[country]['test'].index.to_numpy()
        prediction = pd.DataFrame(data = {'year': x,'prediction': np.squeeze(model.predict(x.reshape(-1, 1)))})
        prediction.set_index('year', inplace = True)
        predictions[country] = prediction
    return(predictions)

In [ ]:
# Exponentieel model door te lineariseren met log en exp
# Buiten dat is het hetzelfde als het voordaande model

def exp_mlr_train(country_dfs):
    models = {}
    for country in country_dfs:
        model = sklearn.linear_model.LinearRegression()
        x = country_dfs[country]['train'].index.to_numpy().reshape(-1, 1)
        # De log in de volgende lijn is hiet het enige verschil
        y = np.log(country_dfs[country]['train'])
        model.fit(x, y)
        models[country] = model
    return(models) #dict van de modellen voor elk land

def exp_mlr_test(country_dfs, models):
    predictions = {}
    for country in country_dfs:
        model = models[country]
        x = country_dfs[country]['test'].index.to_numpy()
        prediction = pd.DataFrame(data = {'year': x,'prediction': np.squeeze(model.predict(x.reshape(-1, 1)))})
        # De volgende lijn is hier het enige verschil
        prediction['prediction'] = np.exp(prediction['prediction'])
        prediction.set_index('year', inplace = True)
        predictions[country] = prediction
    return(predictions)

In [ ]:
# Geintegreerd linear model (linearire fit op de groei)

def i_mlr_train(country_dfs):
    models = {}
    for country in country_dfs:
        model = sklearn.linear_model.LinearRegression()
        x = country_dfs[country]['train'].index.to_numpy().reshape(-1, 1)
        y = country_dfs[country]['train'].pct_change()
        # Het eerste jaar aan data droppen aangezien daar geen vector van te maken is
        x, y = x[1:-1], y[1:-1]
        model.fit(x, y)
        models[country] = model
    return(models) #dict van de modellen voor elk land

def i_mlr_test(country_dfs, models):
    predictions = {}
    for country in country_dfs:
        model = models[country]
        x = country_dfs[country]['test'].index.to_numpy()
        prediction = pd.DataFrame(data = {'year': x,'prediction': np.squeeze(model.predict(x.reshape(-1, 1)))})
        prediction.set_index('year', inplace = True)

        # Hier wordt de laatste waarde uit de train split gebruikt om de groei om te zetten naar wat we proberen te voorspellen
        prediction.loc[prediction.index[0], 'prediction'] = country_dfs['NLD']['train'].iloc[-1] * (1 + prediction.loc[prediction.index[0], 'prediction'])
        for i in prediction.index[1:]:
            prediction.loc[i, 'prediction'] = (1 + prediction.loc[i, 'prediction']) * prediction.loc[i-1, 'prediction']
        predictions[country] = prediction
    return(predictions)

In [9]:
file = open("country_dfs.pkl",'rb')
country_dfs = pickle.load(file)
country_dfs['NLD']['train'].head()

year
1950     93998.617188
1951     96876.679688
1952    106561.078125
1953    115033.203125
1954    118803.335938
Name: rgdpe, dtype: float64

In [234]:
mlr_models = i_mlr_train(country_dfs)
i_mlr_test(country_dfs, mlr_models)['NLD']

,prediction
year,
2000,7.511104e+05
2001,7.716216e+05
2002,7.922449e+05
2003,8.129595e+05
2004,8.337436e+05
2005,8.545750e+05
2006,8.754308e+05
2007,8.962872e+05
2008,9.171202e+05
